In [ ]:

import os, sys, glob, shutil, time, json
import numpy as np, pandas as pd
t0=time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:7.0f}с] {m}", flush=True)
base=os.path.dirname(glob.glob("/kaggle/input/**/items_human.parquet", recursive=True)[0])
llm=os.path.dirname(glob.glob("/kaggle/input/**/llm_pairs_sel.parquet", recursive=True)[0])
os.makedirs("/kaggle/working/src",exist_ok=True)
for p in glob.glob(base+"/*.py"): shutil.copy(p,"/kaggle/working/src/")
open("/kaggle/working/src/__init__.py","a").close()
os.makedirs("/kaggle/working/models",exist_ok=True)
shutil.copy(base+"/anti_words.json","/kaggle/working/models/anti_words.json")
os.chdir("/kaggle/working"); sys.path.insert(0,"/kaggle/working")
from src.attr_features import parse, compare, FEATURE_NAMES
from src.name_features import parse_name, compare_names, build_idf, NAME_FEATURE_NAMES
from src.string_features import compare_strings, STRING_FEATURE_NAMES
from src.neighbour_features import build as nb_build, compare as nb_compare, NEIGHBOUR_FEATURE_NAMES
from src.brand_features import colours, canonical, compare_brands, compare_colours, BRAND_FEATURE_NAMES
from src.hybrid import product_disjoint_pair_masks
from src.metrics import macro_pr_auc
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import average_precision_score

PAIRS=900_000
pairs=pd.read_parquet(llm+"/llm_pairs_sel.parquet").head(PAIRS)
items=pd.read_parquet(llm+"/llm_items_sel.parquet")
need=set(pd.unique(np.concatenate([pairs["id1"].to_numpy(),pairs["id2"].to_numpy()])).tolist())
items=items[items["id"].isin(need)].reset_index(drop=True)
log(f"LLM: пар {len(pairs):,}, карточек {len(items):,}, доля+ {pairs['label'].mean():.3f}")

ALL=FEATURE_NAMES+NAME_FEATURE_NAMES+STRING_FEATURE_NAMES+NEIGHBOUR_FEATURE_NAMES+BRAND_FEATURE_NAMES
def featurize(items, pairs, tag):
    ID=items["id"].to_numpy(); NAME=items["name"].astype(str).tolist(); ATTR=items["attributes"].tolist()
    cards={int(i):parse(n,a,name=n) for i,n,a in zip(ID,NAME,ATTR)}
    names={int(i):parse_name(n) for i,n in zip(ID,NAME)}
    idf,avg=build_idf(list(names.values()))
    cols={int(i):colours(n+" "+str(a)) for i,n,a in zip(ID,NAME,ATTR)}
    brand={int(i):frozenset(x for x in (canonical(v) for v in c.slots.get("brand",())) if x)
           for i,c in cards.items()}
    log(f"{tag}: карточки разобраны")
    profile=nb_build(items[["id","name","category"]]); log(f"{tag}: окрестности готовы")
    sim_of={}
    for cat,g in items.groupby("category",sort=False):
        M=TfidfVectorizer(min_df=1,sublinear_tf=True).fit_transform(g["name"].astype(str).tolist())
        sim_of[cat]=(M,{int(x):r for r,x in enumerate(g["id"].to_numpy())})
    cat_of=dict(zip(ID.tolist(),items["category"].astype(str).tolist()))
    POS={int(x):r for r,x in enumerate(ID)}
    def sim(a,b):
        ca=cat_of.get(a)
        if ca is None or ca!=cat_of.get(b): return 0.0
        M,pos=sim_of[ca]
        if a not in pos or b not in pos: return 0.0
        return float((M[pos[a]]@M[pos[b]].T).toarray()[0,0])
    def row(a,b):
        d=compare(cards[a],cards[b]); d.update(compare_names(names[a],names[b],idf,avg))
        d.update(compare_strings(NAME[POS[a]],NAME[POS[b]]))
        d.update(nb_compare(a,b,sim(a,b),profile))
        d.update(compare_brands(brand[a],brand[b],{})); d.update(compare_colours(cols[a],cols[b]))
        return [d[k] for k in ALL]
    t=time.perf_counter()
    X=np.array([row(int(a),int(b)) for a,b in zip(pairs["id1"],pairs["id2"])],dtype=np.float32)
    log(f"{tag}: {len(pairs):,} пар за {time.perf_counter()-t:.0f}с")
    return X

Xllm=featurize(items,pairs,"LLM"); np.save("/kaggle/working/features_llm.npy",Xllm)
yllm=pairs["label"].to_numpy(np.int8)
del items; import gc; gc.collect()

hitems=pd.read_parquet(base+"/items_human.parquet")
hm=pd.read_parquet(base+"/matches.parquet",columns=["id1","id2","target"])
evalp=pd.read_parquet(base+"/eval_pairs.parquet")
prev=os.path.dirname(glob.glob("/kaggle/input/**/features_new.npy",recursive=True)[0])
Xh=np.load(prev+"/features_new.npy"); Xe=np.load(prev+"/features_eval.npy")
log("человеческие признаки взяты из прошлого ядра")

y=hm["target"].to_numpy(np.int8)
cat_of=dict(zip(hitems["id"],hitems["category"].astype(str)))
cp=hm["id1"].map(cat_of).astype(str).to_numpy()
tm,vm=product_disjoint_pair_masks(hm["id1"].to_numpy(),hm["id2"].to_numpy(),0,3)
tr,va=np.flatnonzero(tm),np.flatnonzero(vm)
ye=evalp["target"].to_numpy(np.int8); ce=evalp["category"].astype(str).to_numpy()
def macro_eval(p): return float(np.mean([average_precision_score(ye[ce==k],p[ce==k])
    for k in np.unique(ce) if len(np.unique(ye[ce==k]))>1]))

log("обучение")
res={}
for tag, Xt, yt in (("только ручные", Xh[tr], y[tr]),
                    ("только LLM", Xllm, yllm),
                    ("LLM + ручные", np.vstack([Xllm, Xh[tr]]), np.concatenate([yllm, y[tr]]))):
    clf=HistGradientBoostingClassifier(max_iter=400,learning_rate=0.08,random_state=0)
    clf.fit(Xt,yt)
    p=clf.predict_proba(Xh[va])[:,1]; pe=clf.predict_proba(Xe)[:,1]
    np.save(f"/kaggle/working/pred_{tag.split()[0]}.npy",pe)
    log(f"{tag:<16} holdout {macro_pr_auc(y[va],p,cp[va])[0]:.6f}   линейка {macro_eval(pe):.6f}")
log("готово")
